[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ai-agents-certified/notebooks/day-08-streaming-and-memory.ipynb#scrollTo=10a2b3c4)

---
# Day 8 · Streaming and Long-Term Memory Store
**certified-journeys / ai-agents-certified** · Day 8 · Streaming & Persistence

> **Goal for today:** Build an agent that streams individual tokens to the terminal as they arrive, shows which graph node is currently executing, and stores a user preference in `InMemoryStore` that survives across conversation turns.


In [ ]:
%pip install -q langgraph langchain-core langchain-openai python-dotenv


## Step 1 · LangGraph Streaming Modes Overview

LangGraph offers two orthogonal streaming modes that can be combined:

| Mode | What it yields | Use for |
|------|---------------|--------|
| `stream_mode="messages"` | `(AIMessageChunk, metadata)` tuples — one per token | Token-level terminal streaming |
| `stream_mode="updates"` | `{node_name: state_delta}` dicts | Node-level tracing / progress UI |
| `astream_events` | Typed event dicts with `event`, `name`, `data` | Fine-grained filtering of specific event types |

**Combining modes:** pass a list — `graph.astream(state, stream_mode=["messages", "updates"])` — and LangGraph multiplexes both streams into a single async generator.

The pattern for a rich terminal UI:
1. `updates` → print node name when it starts
2. `messages` → print each token chunk inline


In [ ]:
import os, asyncio, sys
from typing import Annotated
from langchain_core.messages import HumanMessage, AIMessage, AIMessageChunk, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from typing import TypedDict

os.environ.setdefault("OPENAI_API_KEY", "sk-placeholder")

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

def build_simple_agent(llm):
    """Single-node agent that calls the LLM on every turn."""
    def agent_node(state: ChatState) -> dict:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    builder = StateGraph(ChatState)
    builder.add_node("agent", agent_node)
    builder.set_entry_point("agent")
    builder.add_edge("agent", END)
    return builder.compile()

print("Simple single-node agent graph defined")
print("State schema:", list(ChatState.__annotations__.keys()))


**What just happened?**
- We built the simplest possible LangGraph agent: one node, one LLM call, one edge to END. This is the substrate we'll add streaming and memory to.
- The `add_messages` reducer on the `messages` field means each turn's response is **appended** to the history, not overwritten — conversation context accumulates automatically.
- Importantly, this node calls `llm.invoke()` (synchronous). For streaming we'll switch to `astream_events` or pass the LLM through the graph's own streaming pipeline.


## Step 2 · Token-Level Streaming with `stream_mode="messages"`

`stream_mode="messages"` yields `(chunk, metadata)` tuples where `chunk` is an `AIMessageChunk`. Print `chunk.content` without a newline to simulate real-time token streaming.

The `metadata` dict includes `langgraph_node` — the name of the graph node that produced this chunk. This lets you label which agent is speaking even in multi-agent graphs.


In [ ]:
async def stream_tokens(graph, user_input: str):
    """Stream tokens from the agent to stdout, printing node labels."""
    config = {"configurable": {"thread_id": "demo-thread"}}
    state = {"messages": [HumanMessage(content=user_input)]}

    current_node = None
    print(f"User: {user_input}")

    # stream_mode="messages" yields (AIMessageChunk, metadata) pairs
    async for chunk, metadata in graph.astream(
        state,
        config=config,
        stream_mode="messages",
    ):
        node = metadata.get("langgraph_node", "")

        # Print node header when we enter a new node
        if node != current_node:
            if current_node is not None:
                print()  # newline after previous node's tokens
            print(f"\n[{node}] ", end="", flush=True)
            current_node = node

        # Print the token chunk inline (no newline)
        if isinstance(chunk, AIMessageChunk) and chunk.content:
            print(chunk.content, end="", flush=True)

    print()  # final newline
    print("-" * 40)


# Demonstrate the function signature with a mock (avoids needing a real API key)
import inspect
print("stream_tokens signature:")
print(inspect.signature(stream_tokens))
print("\nTo run with a real model:")
print("  llm = ChatOpenAI(model='gpt-4o-mini', streaming=True)")
print("  graph = build_simple_agent(llm)")
print("  await stream_tokens(graph, 'What is LangGraph?')")


**What just happened?**
- **`astream` with `stream_mode="messages"`** is the async counterpart to `stream`. In Colab you can run it with `await` inside an async cell or wrap with `asyncio.run()`.
- **`metadata["langgraph_node"]`** is automatically populated by LangGraph — you don't need to instrument your nodes.
- `flush=True` on `print` is critical: without it, Python's stdout buffer holds tokens and releases them in batches, destroying the streaming UX.


## Step 3 · Node-Level Streaming with `stream_mode="updates"`

While `messages` streams individual tokens, `updates` streams entire state deltas at the node boundary. Use `updates` to build a progress indicator:

```
[supervisor] deciding route…
[researcher] running…
[writer] running…
Done.
```

Mixing both modes in a single `astream` call lets you build a terminal UI with both progress labels and streaming tokens.


In [ ]:
async def stream_with_node_labels(graph, user_input: str):
    """Combine node-level progress labels with token streaming."""
    state = {"messages": [HumanMessage(content=user_input)]}

    print(f"User: {user_input}\n")

    # Pass a list to get both stream types multiplexed
    async for event_type, payload in graph.astream(
        state,
        stream_mode=["messages", "updates"],  # both modes at once
    ):
        if event_type == "updates":
            # payload is {node_name: state_delta}
            for node_name in payload:
                print(f"\n▶ [{node_name}] executing…", flush=True)

        elif event_type == "messages":
            # payload is (AIMessageChunk, metadata)
            chunk, metadata = payload
            if isinstance(chunk, AIMessageChunk) and chunk.content:
                print(chunk.content, end="", flush=True)

    print("\n\n✓ Complete")


# Verify the event_type branching logic with a simulated event sequence
simulated_events = [
    ("updates", {"agent": {"messages": []}}),
    ("messages", (AIMessageChunk(content="Hello"), {"langgraph_node": "agent"})),
    ("messages", (AIMessageChunk(content=" world"), {"langgraph_node": "agent"})),
]

print("Simulated dual-mode event trace:")
for event_type, payload in simulated_events:
    if event_type == "updates":
        for node_name in payload:
            print(f"\n▶ [{node_name}] executing…")
    elif event_type == "messages":
        chunk, _ = payload
        print(chunk.content, end="", flush=True)
print("\n\n✓ Simulation complete")


**What just happened?**
- When you pass a **list** to `stream_mode`, LangGraph multiplexes both streams into a single generator that yields `(event_type, payload)` tuples.
- Branching on `event_type` keeps the handler clean — node progress goes to one branch, token streaming to the other.
- The simulation confirms the event shape without needing an API call — test the event parsing logic independently from the LLM.


## Step 4 · Filtering Events with `astream_events`

`astream_events` provides the finest-grained control: every LangChain callback event is surfaced as a typed dict. You can filter for specific event names to ignore noise from tools or sub-chains you don't care about.

**Key event types:**

| Event name | Fires when |
|-----------|------------|
| `on_chat_model_stream` | LLM outputs a token |
| `on_tool_start` | A tool begins execution |
| `on_tool_end` | A tool finishes |
| `on_chain_start` | A node/chain starts |
| `on_chain_end` | A node/chain finishes |


In [ ]:
async def stream_events_filtered(graph, user_input: str):
    """Use astream_events to filter for LLM tokens only — ignore tool noise."""
    state = {"messages": [HumanMessage(content=user_input)]}

    print(f"User: {user_input}\nAgent: ", end="", flush=True)

    # version="v2" is the current stable API for astream_events
    async for event in graph.astream_events(state, version="v2"):
        event_name = event["event"]

        # Filter: only process LLM token events
        if event_name == "on_chat_model_stream":
            chunk = event["data"]["chunk"]       # AIMessageChunk
            if chunk.content:                     # skip empty tool-call chunks
                print(chunk.content, end="", flush=True)

        # Optional: log tool calls for debugging
        elif event_name == "on_tool_start":
            tool_name = event["name"]
            print(f"\n  [tool:{tool_name}] starting…", flush=True)

    print()  # newline after streaming


# Show what a real astream_events event looks like
example_event = {
    "event": "on_chat_model_stream",
    "name": "ChatOpenAI",
    "run_id": "abc-123",
    "data": {
        "chunk": AIMessageChunk(content="Hello")
    },
    "metadata": {"langgraph_node": "agent"}
}

print("Example astream_events event structure:")
print(f"  event: {example_event['event']}")
print(f"  name:  {example_event['name']}")
print(f"  data.chunk.content: '{example_event['data']['chunk'].content}'")
print(f"  metadata.langgraph_node: '{example_event['metadata']['langgraph_node']}'")
print("\nFilter pattern: event['event'] == 'on_chat_model_stream'")


**What just happened?**
- **`version="v2"`** is mandatory — v1 had breaking changes and is deprecated. Always specify it explicitly.
- Filtering on `event_name == "on_chat_model_stream"` isolates LLM tokens from all other events (tool calls, chain starts, etc.) — no more noise in the terminal output.
- **`if chunk.content`** skips empty chunks: LLMs often emit an empty chunk at the start and end of a streaming response as bookends.


## Step 5 · Long-Term Memory with `InMemoryStore`

LangGraph's memory hierarchy:

| Layer | What it stores | Lifespan |
|-------|---------------|----------|
| `messages` in state | Conversation turns | One thread (wiped on new thread_id) |
| **`InMemoryStore`** | Arbitrary key-value data | **Cross-thread, in-process** |
| `SqliteSaver` / `AsyncSqliteSaver` | State snapshots | Cross-session, on disk |

`InMemoryStore` uses a **namespace tuple** as the key space. A common pattern: `("user_prefs", user_id)` → store user preferences that survive across multiple conversation threads.


In [ ]:
from langgraph.store.memory import InMemoryStore

# Create a store — this survives across graph invocations in the same process
store = InMemoryStore()

USER_ID = "user-42"
PREFS_NAMESPACE = ("user_prefs", USER_ID)  # namespace is a tuple of strings

# ── Store a preference ────────────────────────────────────────────────────────
store.put(
    namespace=PREFS_NAMESPACE,
    key="response_style",           # key within the namespace
    value={"style": "concise", "bullet_points": True}  # any JSON-serializable dict
)

print("Stored preference:")
print(f"  namespace: {PREFS_NAMESPACE}")
print(f"  key:       'response_style'")
print(f"  value:     {{style: 'concise', bullet_points: True}}")

# ── Retrieve the preference ───────────────────────────────────────────────────
item = store.get(namespace=PREFS_NAMESPACE, key="response_style")
print(f"\nRetrieved: {item.value}")
print(f"Item type: {type(item)}")
print(f"Item key:  {item.key}")

# ── Search all items in a namespace ──────────────────────────────────────────
store.put(PREFS_NAMESPACE, "language", {"lang": "en"})
store.put(PREFS_NAMESPACE, "timezone", {"tz": "UTC"})

all_items = list(store.search(PREFS_NAMESPACE))
print(f"\nAll items in namespace {PREFS_NAMESPACE}:")
for it in all_items:
    print(f"  {it.key}: {it.value}")


**What just happened?**
- **`InMemoryStore`** uses `(namespace_tuple, key)` addressing — the namespace provides a logical grouping (user, session, agent) and the key identifies the specific record.
- `store.get()` returns an `Item` object with `.value`, `.key`, and `.namespace` attributes — not the raw value directly.
- **`store.search(namespace)`** returns all items in that namespace as a list. This is how an agent would load all preferences for a user at the start of a conversation.


## Step 6 · Wiring the Store into the Agent Graph

To inject the store into a graph node, compile the graph with `store=` and accept it via the node's `RunnableConfig`. LangGraph passes the store through `config["store"]` automatically.

Pattern:
```python
graph = builder.compile(store=store)
# In the node:
def my_node(state, config, *, store):
    prefs = store.get(("user_prefs", user_id), "response_style")
```


In [ ]:
from langgraph.graph import StateGraph, END
from langchain_core.runnables import RunnableConfig

def build_memory_agent(llm, store: InMemoryStore):
    """Agent that reads and writes user preferences via InMemoryStore."""

    def agent_node(state: ChatState, config: RunnableConfig) -> dict:
        user_id = config.get("configurable", {}).get("user_id", "default")
        namespace = ("user_prefs", user_id)

        # Load preferences — returns None if not set yet
        pref_item = store.get(namespace, "response_style")
        style_hint = ""
        if pref_item:
            style = pref_item.value.get("style", "normal")
            use_bullets = pref_item.value.get("bullet_points", False)
            style_hint = f" Respond in a {style} style{', using bullet points' if use_bullets else ''}."

        # Build system prompt incorporating the stored preference
        system = SystemMessage(content=f"You are a helpful assistant.{style_hint}")
        messages = [system] + list(state["messages"])

        # Simulate LLM response (replace with llm.invoke(messages) for real runs)
        response_text = f"[Agent response — style hint applied: '{style_hint.strip()}']" if style_hint else "[Agent response — no style preference set]"
        response = AIMessage(content=response_text)

        # Detect preference commands in the user message and persist them
        last_user_msg = state["messages"][-1].content if state["messages"] else ""
        if "prefer concise" in last_user_msg.lower():
            store.put(namespace, "response_style", {"style": "concise", "bullet_points": True})
            print(f"  [Memory] Saved preference 'concise + bullets' for user {user_id}")
        elif "prefer detailed" in last_user_msg.lower():
            store.put(namespace, "response_style", {"style": "detailed", "bullet_points": False})
            print(f"  [Memory] Saved preference 'detailed' for user {user_id}")

        return {"messages": [response]}

    builder = StateGraph(ChatState)
    builder.add_node("agent", agent_node)
    builder.set_entry_point("agent")
    builder.add_edge("agent", END)
    return builder.compile(store=store)   # pass store at compile time


# Demonstration: two turns — preference set on turn 1, applied on turn 2
fresh_store = InMemoryStore()
mem_graph = build_memory_agent(llm=None, store=fresh_store)  # llm=None, using simulated response

config_user_42 = {"configurable": {"user_id": "user-42", "thread_id": "t1"}}

# Turn 1: user sets a preference
print("=== Turn 1: Set preference ===")
result1 = mem_graph.invoke(
    {"messages": [HumanMessage(content="I prefer concise answers with bullets.")]},
    config=config_user_42
)
print(f"Agent: {result1['messages'][-1].content}")

# Turn 2: new thread_id — fresh conversation, but store persists
config_user_42_t2 = {"configurable": {"user_id": "user-42", "thread_id": "t2"}}
print("\n=== Turn 2: New thread, preference retrieved ===")
result2 = mem_graph.invoke(
    {"messages": [HumanMessage(content="What is LangGraph?")]},
    config=config_user_42_t2
)
print(f"Agent: {result2['messages'][-1].content}")


**What just happened?**
- **`builder.compile(store=store)`** wires the store into the graph. The node accesses it via its function signature — LangGraph injects it automatically when the node accepts a `config` parameter.
- **`config["configurable"]["user_id"]`** is the idiomatic way to pass per-request metadata (user ID, session flags) without polluting graph state.
- The key insight: **`thread_id` changed between turns but `user_id` stayed the same** — the InMemoryStore returned the preference from Turn 1 even though Turn 2 is a completely fresh conversation thread. This is the definition of long-term memory.


## Step 7 · Full Streaming + Memory Agent

Combining everything: a single agent that streams tokens, shows node labels, and retrieves stored preferences before generating its response.


In [ ]:
async def run_streaming_memory_demo():
    """End-to-end demo: streaming + preference persistence."""
    demo_store = InMemoryStore()

    # Pre-seed a preference
    demo_store.put(
        ("user_prefs", "alice"),
        "response_style",
        {"style": "concise", "bullet_points": True}
    )
    print("[Store] Pre-seeded 'concise + bullets' preference for alice")

    # Simulate streaming event sequence that a real graph would emit
    simulated_stream = [
        ("updates", {"agent": {"messages": []}}),
        ("messages", (AIMessageChunk(content="Here"), {"langgraph_node": "agent"})),
        ("messages", (AIMessageChunk(content=" are"), {"langgraph_node": "agent"})),
        ("messages", (AIMessageChunk(content=" the key points:"), {"langgraph_node": "agent"})),
        ("messages", (AIMessageChunk(content="\n- Point 1"), {"langgraph_node": "agent"})),
        ("messages", (AIMessageChunk(content="\n- Point 2"), {"langgraph_node": "agent"})),
    ]

    # Load and display preference
    pref = demo_store.get(("user_prefs", "alice"), "response_style")
    print(f"[Store] Loaded alice's preference: {pref.value}")
    print()

    current_node = None
    print("User [alice]: What are the benefits of LangGraph?")

    for event_type, payload in simulated_stream:
        if event_type == "updates":
            for node_name in payload:
                current_node = node_name
                print(f"\n▶ [{node_name}] ", end="", flush=True)
        elif event_type == "messages":
            chunk, _ = payload
            if chunk.content:
                print(chunk.content, end="", flush=True)

    print("\n\n✓ Response complete")

    # Verify store still holds alice's preference
    all_alice_prefs = list(demo_store.search(("user_prefs", "alice")))
    print(f"[Store] alice's stored preferences: {[i.key for i in all_alice_prefs]}")


# Run the demo
asyncio.run(run_streaming_memory_demo()) if not asyncio.get_event_loop().is_running() \
    else await run_streaming_memory_demo()


**What just happened?**
- The demo confirms all three layers working together: store retrieval before the node runs, node-label printing from `updates` events, and token streaming from `messages` events.
- **`asyncio.run()` vs `await`** — Colab's kernel runs an event loop, so you need `await` directly. In a standalone script, use `asyncio.run()`. The ternary handles both.
- After the demo, `store.search()` confirms the preference survived the entire interaction — it wasn't wiped by the graph invocation.


In [ ]:
# Challenge: Streaming agent with preference-aware system prompt
#
# Build an agent that:
# 1. Reads 'response_style' from InMemoryStore on every turn.
# 2. Detects "set preference: <style>" in user messages and writes it to the store.
# 3. Streams every LLM token using astream with stream_mode="messages".
# 4. Prints "▶ [node_name]" before the first token from each new node.
# 5. Runs a 2-turn conversation:
#    Turn 1: "set preference: bullet_points=True, style=concise"
#    Turn 2: "Explain what a StateGraph is" — should use the saved style.
#
# Scaffold:
# store = InMemoryStore()
# llm = ChatOpenAI(model="gpt-4o-mini", streaming=True)  # set OPENAI_API_KEY
#
# def parse_preference(text: str) -> dict | None:
#     """Return a preference dict if the message is a preference command, else None."""
#     # Your parsing logic here
#     pass
#
# def pref_aware_node(state: ChatState, config: RunnableConfig) -> dict:
#     # 1. Get user_id from config
#     # 2. Load preference from store
#     # 3. Build system prompt with style hint
#     # 4. Detect and save new preferences
#     # 5. Call LLM and return response
#     pass
#
# async def run():
#     # Build graph, run 2 turns with streaming
#     pass
#
# await run()

# Your solution here


---
## Day 8 key concepts recap

| Concept | What to remember |
|---|---|
| `stream_mode="messages"` | Yields `(AIMessageChunk, metadata)` — one per token |
| `stream_mode="updates"` | Yields `{node_name: state_delta}` — one per node execution |
| Combining modes | Pass a list; LangGraph yields `(event_type, payload)` tuples |
| `astream_events(version="v2")` | Finest-grained; filter on `event['event'] == 'on_chat_model_stream'` |
| `InMemoryStore` | `store.put(namespace, key, value)` / `store.get(namespace, key)` |
| Cross-thread memory | Store survives thread_id changes; use `user_id` as namespace key |
| `compile(store=store)` | Injects store into nodes that accept `config` parameter |

> **Tip:** Use `stream_mode='messages'` for token-level streaming and `stream_mode='updates'` for node-level updates. Mixing both in `astream` lets you build a rich terminal UI without any extra dependencies.

---
## What's next
**Day 9** → Review and Patterns — rebuild the supervisor graph from memory, write a cheat sheet, and audit your Day 7 graph for edge cases.

Mark Day 8 complete in your [tracker](../index.html).
